In [28]:
from sklearn.datasets import load_breast_cancer

In [29]:
data=load_breast_cancer()

In [30]:
X=data.data
y=data.target

In [31]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [32]:
from catboost import CatBoostClassifier

In [33]:
cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=5,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose= False
)

In [34]:
cat_model.fit(X_train,y_train)

CatBoostClassifier(depth=5, eval_metric='AUC', iterations=100, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=False)

In [35]:
y_pred = cat_model.predict(X_test)
y_proba= cat_model.predict_proba(X_test)[:,1]

In [36]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score

print("Accuracy",accuracy_score(y_test,y_pred))

print("\nConfusion matrix")
print(confusion_matrix(y_test,y_pred))

print("\nClassification Report")
print(classification_report(y_test,y_pred))

print(roc_auc_score(y_test,y_proba))



Accuracy 0.9736842105263158

Confusion matrix
[[41  2]
 [ 1 70]]

Classification Report
              precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

0.9967245332459875


In [39]:
from sklearn.model_selection import GridSearchCV

cat_model=CatBoostClassifier(
    loss_function="Logloss",
    eval_metric= "AUC",
    random_seed=42,
    verbose=False
)

param_grid = {
    "iterations":[100,200],
    "depth":[4,5,6],
    "learning_rate":[0.05,0.1],
    "l2_leaf_reg":[3,5,10]
  }

grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train,y_train)

print("Best Parameters",grid_search.best_params_)
print("Best Estimator",grid_search.best_score_)

Best Parameters {'depth': 6, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.1}
Best Estimator 0.9927708816573981


In [42]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]

print("Accuracy",accuracy_score(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

print("/nClassification Report")
print(classification_report(y_test,y_pred))

print(roc_auc_score(y_test,y_proba))

Accuracy 0.9649122807017544

Confusion Matrix
[[40  3]
 [ 1 70]]
/nClassification Report
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

0.9941041598427777


In [43]:
from sklearn.model_selection import train_test_split

X_train_cb, X_val_cb, y_train_cb, y_val_cb = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [44]:
cat_model_es = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

cat_model_es.fit(
    X_train_cb,
    y_train_cb,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=50
)

CatBoostClassifier(depth=6, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [45]:
print("Best iteration:", cat_model_es.get_best_iteration())
print("Best score:", cat_model_es.get_best_score())

Best iteration: 20
Best score: {'learn': {'Logloss': 0.022302643283382877}, 'validation': {'Logloss': 0.14273731658159652, 'AUC': 0.9865841073271414}}


In [46]:
y_pred_es = cat_model_es.predict(X_test)
y_prob_es = cat_model_es.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_es))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_es))

print("\nClassification Report")
print(classification_report(y_test, y_pred_es))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_es))

Accuracy: 0.956140350877193

Confusion Matrix
[[40  3]
 [ 2 69]]

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

ROC-AUC: 0.9941041598427776
